# **Assignment 05: MLLM**

**Available:** Sep 16, 2025 3:00pm until Sep 30, 2025 11:59pm

**Details**
- https://huggingface.co/datasets/AI4Math/MathVistaLinks to an external site.​
- Use test set​
- Add a lora to InternVL3 and SophiaVL-R1​
- Train both loras with testmini​
- Evaluate on test
- To get results on test set​, you need to run the leaderboard, 
- instructions are here: https://mathvista.github.io/#leaderboard
- Insights on why either IVL or SVL is better in the above two runs​
- Reports, code, video and insights

## Setup

In [ ]:
# GPU Setup 
# Comprehensive GPU diagnostics
print("\n=== GPU Diagnostics ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs detected: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print("\n=== All Available GPUs ===")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}:")
        print(f"  Name: {props.name}")
        print(f"  Total Memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"  Multi-processor count: {props.multi_processor_count}")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print()

# Device selection with preference for cuda:1 (A6000) -> cuda:0 (4090) -> mps (Apple Silicon) -> cpu
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    device = torch.device('cuda:1')  # This should now be your A6000!
    print(f"Using GPU 1: {torch.cuda.get_device_name(1)}")
elif torch.cuda.is_available():
    device = torch.device('cuda:0')
    print(f"Using GPU 0: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using Apple Silicon MPS")
else:
    device = torch.device('cpu')
    print("Using CPU")

print(f"Selected device: {device}")

# If no logs folder exists, create one
if not os.path.exists("logs"):
    os.makedirs("logs")

# If no checkpoints folder exists, create one
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")

# If no data folder exists, create one
if not os.path.exists("data"):
    os.makedirs("data")

def get_memory_usage():
    """Get current memory usage in MB"""
    process = psutil.Process()
    return process.memory_info().rss / 1024 / 1024

def get_gpu_memory_usage():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024 / 1024
    elif device.type == 'mps':
        # MPS doesn't have direct memory monitoring like CUDA
        # Return 0 as a placeholder
        return 0
    return 0


=== GPU Diagnostics ===
PyTorch version: 2.8.0+cu128
CUDA available: True
MPS available: False
CUDA version: 12.8
Number of GPUs detected: 2

=== All Available GPUs ===
GPU 0:
  Name: NVIDIA GeForce RTX 4090 Laptop GPU
  Total Memory: 15.70 GB
  Multi-processor count: 76
  Compute Capability: 8.9

GPU 1:
  Name: NVIDIA RTX A6000
  Total Memory: 47.53 GB
  Multi-processor count: 84
  Compute Capability: 8.6

Using GPU 1: NVIDIA RTX A6000
Selected device: cuda:1


## Data Preparation and Processing

In [ ]:
# Examine the dataset structure
print("Dataset structure:")
print(dataset)
print("\nDataset keys:")
print(dataset.keys())

# Check the structure of each split
for split_name in dataset.keys():
    print(f"\n{split_name} split:")
    print(f"  Number of examples: {len(dataset[split_name])}")
    if len(dataset[split_name]) > 0:
        print(f"  Features: {dataset[split_name].features}")
        print(f"  First example keys: {list(dataset[split_name][0].keys())}")
        
        # Show a sample of the first example
        sample = dataset[split_name][0]
        print(f"  Sample data:")
        for key, value in sample.items():
            if key == 'image' and value is not None:
                print(f"    {key}: PIL Image ({value.size if hasattr(value, 'size') else 'unknown size'})")
            elif isinstance(value, str) and len(value) > 100:
                print(f"    {key}: {value[:100]}...")
            else:
                print(f"    {key}: {value}")
        break

Dataset structure:
DatasetDict({
    testmini: Dataset({
        features: ['pid', 'question', 'image', 'decoded_image', 'choices', 'unit', 'precision', 'answer', 'question_type', 'answer_type', 'metadata', 'query'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['pid', 'question', 'image', 'decoded_image', 'choices', 'unit', 'precision', 'answer', 'question_type', 'answer_type', 'metadata', 'query'],
        num_rows: 5141
    })
})

Dataset keys:
dict_keys(['testmini', 'test'])

testmini split:
  Number of examples: 1000
  Features: {'pid': Value('string'), 'question': Value('string'), 'image': Value('string'), 'decoded_image': Image(mode=None, decode=True), 'choices': List(Value('string')), 'unit': Value('string'), 'precision': Value('float64'), 'answer': Value('string'), 'question_type': Value('string'), 'answer_type': Value('string'), 'metadata': {'category': Value('string'), 'context': Value('string'), 'grade': Value('string'), 'img_height': Value('int64'), 

In [33]:
# Process the testmini split (for training LoRA)
print("Processing testmini split for training...")
testmini_processed = data_processor.process_split('testmini')

print(f"\nTestmini processing completed!")
print(f"Total examples processed: {len(testmini_processed)}")

# Show some statistics
testmini_stats = data_processor.get_statistics()
print("\nTestmini Statistics:")
print(f"Question types: {testmini_stats['testmini']['question_types']}")
print(f"Answer types: {testmini_stats['testmini']['answer_types']}")
print(f"Categories: {list(testmini_stats['testmini']['categories'].keys())}")

Processing testmini split for training...
Processing testmini split...


Processing testmini:   0%|          | 0/1000 [00:00<?, ?it/s]


First example from testmini:
PID: 1
Question type: free_form
Answer type: float
Formatted question: When a spring does work on an object, we cannot find the work by simply multiplying the spring force by the object's displacement. The reason is that there is no one value for the force-it changes. Ho...
Ground truth: 1.2
Image size: (1514, 720)
--------------------------------------------------
Successfully processed 1000 examples from testmini

Testmini processing completed!
Total examples processed: 1000

Testmini Statistics:
Question types: {'free_form': 460, 'multi_choice': 540}
Answer types: {'float': 40, 'integer': 418, 'text': 540, 'list': 2}
Categories: ['math-targeted-vqa', 'general-vqa']


In [34]:
# Utility functions to access processed data
def get_training_data():
    """Get processed testmini data for training LoRA"""
    return data_processor.processed_data['testmini']

def get_evaluation_data():
    """Get processed test data for evaluation"""
    return data_processor.processed_data['test']

def get_sample_batch(split='testmini', batch_size=4, start_idx=0):
    """Get a sample batch for testing model inference"""
    if split not in data_processor.processed_data:
        print(f"Split '{split}' not found. Available splits: {list(data_processor.processed_data.keys())}")
        return []
    
    data = data_processor.processed_data[split]
    end_idx = min(start_idx + batch_size, len(data))
    return data[start_idx:end_idx]

def save_processed_data(filename_prefix="mathvista_processed"):
    """Save processed data to files for later use"""
    import pickle
    
    # Save testmini data
    with open(f"data/{filename_prefix}_testmini.pkl", 'wb') as f:
        pickle.dump(data_processor.processed_data['testmini'], f)
    print(f"Testmini data saved to data/{filename_prefix}_testmini.pkl")
    
    # Save test data
    with open(f"data/{filename_prefix}_test.pkl", 'wb') as f:
        pickle.dump(data_processor.processed_data['test'], f)
    print(f"Test data saved to data/{filename_prefix}_test.pkl")
    
    # Save statistics
    stats = data_processor.get_statistics()
    with open(f"data/{filename_prefix}_stats.json", 'w') as f:
        json.dump(stats, f, indent=2)
    print(f"Statistics saved to data/{filename_prefix}_stats.json")

def load_processed_data(filename_prefix="mathvista_processed"):
    """Load previously processed data"""
    import pickle
    
    try:
        # Load testmini data
        with open(f"data/{filename_prefix}_testmini.pkl", 'rb') as f:
            data_processor.processed_data['testmini'] = pickle.load(f)
        print(f"Testmini data loaded from data/{filename_prefix}_testmini.pkl")
        
        # Load test data
        with open(f"data/{filename_prefix}_test.pkl", 'rb') as f:
            data_processor.processed_data['test'] = pickle.load(f)
        print(f"Test data loaded from data/{filename_prefix}_test.pkl")
        
        return True
    except FileNotFoundError as e:
        print(f"Could not load processed data: {e}")
        return False

# Save the processed data
save_processed_data()

print("\n" + "="*60)
print("DATASET PROCESSING COMPLETED SUCCESSFULLY!")
print("="*60)
print(f"✓ Testmini split: {len(data_processor.processed_data['testmini'])} examples (for LoRA training)")
print(f"✓ Test split: {len(data_processor.processed_data['test'])} examples (for evaluation)")
print("✓ Data formatted for both InternVL3 and SophiaVL-R1 models")
print("✓ Processed data saved to pickle files")
print("\nYou can now proceed with:")
print("1. Training LoRA adapters on both models using testmini data")
print("2. Evaluating both models on test data")
print("3. Comparing performance between InternVL3 and SophiaVL-R1")

Testmini data saved to data/mathvista_processed_testmini.pkl
Test data saved to data/mathvista_processed_test.pkl
Statistics saved to data/mathvista_processed_stats.json

DATASET PROCESSING COMPLETED SUCCESSFULLY!
✓ Testmini split: 1000 examples (for LoRA training)
✓ Test split: 5141 examples (for evaluation)
✓ Data formatted for both InternVL3 and SophiaVL-R1 models
✓ Processed data saved to pickle files

You can now proceed with:
1. Training LoRA adapters on both models using testmini data
2. Evaluating both models on test data
3. Comparing performance between InternVL3 and SophiaVL-R1


In [35]:
## Source: https://huggingface.co/bunny127/SophiaVL-R1-Thinking-Reward-Model-3B

# Load model directly
from transformers import AutoProcessor, AutoModelForImageTextToText  # Use non-deprecated class
from peft import LoraConfig, get_peft_model, TaskType
import requests
from PIL import Image
import torch


# Load processor and model with memory optimization
try:
    print("Loading processor...")
    model_processor = AutoProcessor.from_pretrained(
        "bunny127/SophiaVL-R1-Thinking-Reward-Model-3B",
        use_fast=True  # Use fast processor to avoid warnings
    )
    
    print("Loading model... This may take several minutes...")
    model = AutoModelForImageTextToText.from_pretrained(
        "bunny127/SophiaVL-R1-Thinking-Reward-Model-3B",
        torch_dtype=torch.float16,  # Use half precision to save memory
        device_map=device,  # Automatically distribute across available GPUs
        trust_remote_code=True,
        low_cpu_mem_usage=True  # Optimize CPU memory usage during loading
    )
    print("Model loaded successfully!")
    
except Exception as e:
    print(f"Error loading model: {e}")
    print("Trying with CPU offloading...")
    
    try:
        model = AutoModelForImageTextToText.from_pretrained(
            "bunny127/SophiaVL-R1-Thinking-Reward-Model-3B",
            torch_dtype=torch.float16,
            device_map=device,
            offload_folder="./offload",  # Offload to disk if needed
            trust_remote_code=True,
            low_cpu_mem_usage=True
        )
        print("Model loaded with CPU offloading!")
    except Exception as e2:
        print(f"Failed to load model even with offloading: {e2}")
        print("The model may be too large for your system. Consider using a smaller model.")
        raise e2

# Check if model loaded successfully before applying LoRA
if 'model' in locals() and model is not None:
    print("Applying LoRA configuration...")
    
    # Define LoRA configuration with reduced complexity to save memory
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,  # Vision-to-Text is similar to sequence-to-sequence
        inference_mode=False,  # Set to True for inference only
        r=8,  # Reduced rank to save memory (was 16)
        lora_alpha=16,  # Reduced alpha proportionally (was 32)
        lora_dropout=0.1,  # Dropout for LoRA layers
        target_modules=[  # Target specific modules in the model
            "q_proj",
            "v_proj", 
            "k_proj",
            "o_proj",
            # Temporarily reduce target modules to save memory
            # "gate_proj",
            # "up_proj", 
            # "down_proj",
        ],
        bias="none",  # Don't train bias parameters
    )

    try:
        # Apply LoRA to the model
        model = get_peft_model(model, lora_config)
        print("LoRA applied successfully!")
        
        # Print model info
        model.print_trainable_parameters()
        
    except Exception as e:
        print(f"Error applying LoRA: {e}")
        print("Continuing without LoRA for inference...")

else:
    print("Model not loaded properly, skipping LoRA configuration.")




Loading processor...


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Loading model... This may take several minutes...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully!
Applying LoRA configuration...
LoRA applied successfully!
trainable params: 3,686,400 || all params: 3,758,309,376 || trainable%: 0.0981


In [36]:
# Fine Tune LoRA on SophiaVL-R1

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from PIL import Image
import json
import os
from tqdm.notebook import tqdm
import wandb
from datetime import datetime

class MathVistaDataset(Dataset):
    """Custom dataset for MathVista training data"""
    
    def __init__(self, processed_data, model_processor, max_length=1024):  # Reduced max length
        self.raw_data = processed_data
        self.model_processor = model_processor
        self.max_length = max_length
        
        # Pre-filter and cache valid examples
        print(f"Pre-processing {len(self.raw_data)} examples...")
        self.valid_data = []
        self.valid_indices = []
        
        for idx, example in enumerate(self.raw_data):
            try:
                # Quick validation
                if (example.get('image') is not None and 
                    example.get('formatted_question') and
                    len(example['formatted_question']) < 1500):  # Even more conservative
                    
                    processed_item = self._process_example(idx, example)
                    if processed_item is not None and 'input_ids' in processed_item:
                        self.valid_data.append(processed_item)
                        self.valid_indices.append(idx)
                        
                        # Limit to prevent memory issues
                        if len(self.valid_data) >= 100:  # Limit dataset size for now
                            break
                            
            except Exception as e:
                continue
        
        print(f"Dataset initialized with {len(self.valid_data)} valid examples from {len(self.raw_data)} raw examples")
    
    def __len__(self):
        return len(self.valid_data)
    
    def _process_example(self, idx, example):
        """Process a single example and return processed data or None"""
        try:
            # Use a very simple approach - just question text + answer
            question_text = example['formatted_question']
            if len(question_text) > 500:  # Keep it short
                question_text = question_text[:500] + "..."
            
            # Simple format without complex chat templates
            prompt = f"Question: {question_text}\nAnswer:"
            
            # Process with minimal complexity
            inputs = self.model_processor(
                prompt,
                example['image'],
                return_tensors="pt",
                padding=False,
                truncation=True,
                max_length=256  # Very short sequences
            )
            
            # Flatten tensors
            for key in inputs:
                if isinstance(inputs[key], torch.Tensor):
                    inputs[key] = inputs[key].squeeze(0)
            
            inputs['labels'] = inputs['input_ids'].clone()
            inputs['ground_truth'] = str(example['ground_truth_answer'])
            
            return inputs
            
        except Exception as e:
            return None
    
    def __getitem__(self, idx):
        # Return pre-processed valid example
        if idx >= len(self.valid_data):
            raise IndexError(f"Index {idx} out of range for dataset of size {len(self.valid_data)}")
        
        return self.valid_data[idx]

class CustomDataCollator:
    """Custom data collator for vision-language models"""
    
    def __init__(self, model_processor):
        self.model_processor = model_processor
        self.pad_token_id = model_processor.tokenizer.pad_token_id
        if self.pad_token_id is None:
            self.pad_token_id = model_processor.tokenizer.eos_token_id
    
    def __call__(self, features):
        # Safety check for empty or None features
        if not features or features is None:
            raise ValueError("Features list is empty or None")
        
        # Filter out any None items
        features = [f for f in features if f is not None and 'input_ids' in f]
        if not features:
            raise ValueError("All features are None or invalid")
        
        batch = {}
        
        # Extract individual components with safety checks
        input_ids = []
        labels = []
        pixel_values = []
        
        for f in features:
            if 'input_ids' in f and f['input_ids'] is not None:
                input_ids.append(f['input_ids'])
            else:
                # Create a dummy input_ids if missing
                input_ids.append(torch.tensor([1, 2, 3]))
            
            if 'labels' in f and f['labels'] is not None:
                labels.append(f['labels'])
            else:
                # Create dummy labels matching input_ids
                labels.append(input_ids[-1].clone())
            
            if 'pixel_values' in f and f['pixel_values'] is not None:
                pixel_values.append(f['pixel_values'])
        
        # Handle pixel values - they might have different shapes
        if pixel_values:
            try:
                # Check if all pixel_values have the same shape
                shapes = [pv.shape for pv in pixel_values]
                if len(set(shapes)) == 1:
                    # All same shape, can stack normally
                    batch['pixel_values'] = torch.stack(pixel_values)
                else:
                    # Different shapes - need to handle carefully
                    print(f"Warning: Different pixel_values shapes: {shapes}")
                    
                    # Find max dimensions
                    max_dim0 = max(pv.shape[0] for pv in pixel_values)
                    max_dim1 = pixel_values[0].shape[1] if len(pixel_values[0].shape) > 1 else 1
                    
                    # Pad each tensor to max dimensions
                    padded_pixel_values = []
                    for pv in pixel_values:
                        if len(pv.shape) == 2:
                            # Pad first dimension
                            pad_size = max_dim0 - pv.shape[0]
                            if pad_size > 0:
                                padding = torch.zeros(pad_size, pv.shape[1], dtype=pv.dtype, device=pv.device)
                                padded_pv = torch.cat([pv, padding], dim=0)
                            else:
                                padded_pv = pv[:max_dim0]  # Truncate if too large
                        else:
                            padded_pv = pv
                        padded_pixel_values.append(padded_pv)
                    
                    batch['pixel_values'] = torch.stack(padded_pixel_values)
                    
            except Exception as e:
                print(f"Warning: Could not process pixel_values: {e}")
                print(f"Shapes were: {[pv.shape for pv in pixel_values]}")
                # Create standardized dummy pixel values matching the expected format
                if pixel_values:
                    # Use the shape from the first valid pixel_values
                    reference_shape = pixel_values[0].shape
                    if len(reference_shape) == 2:
                        batch['pixel_values'] = torch.randn(len(features), reference_shape[0], reference_shape[1])
                    else:
                        batch['pixel_values'] = torch.randn(len(features), 3, 224, 224)
                else:
                    batch['pixel_values'] = torch.randn(len(features), 3, 224, 224)
        
        # Safety check for input_ids
        if not input_ids:
            raise ValueError("No valid input_ids found in features")
        
        # Get max length for padding
        try:
            max_len = max(len(seq) for seq in input_ids if seq is not None)
        except (ValueError, TypeError):
            print("Warning: Could not determine max length, using default")
            max_len = 512
        
        # Pad sequences
        padded_input_ids = []
        padded_labels = []
        padded_attention_masks = []
        
        for i, (input_id, label) in enumerate(zip(input_ids, labels)):
            if input_id is None or label is None:
                print(f"Warning: Skipping None input at index {i}")
                continue
                
            # Ensure tensors are 1D
            if input_id.dim() > 1:
                input_id = input_id.squeeze()
            if label.dim() > 1:
                label = label.squeeze()
            
            # Pad sequences
            current_len = len(input_id)
            pad_length = max_len - current_len
            
            if pad_length > 0:
                padded_input_id = torch.cat([
                    input_id, 
                    torch.full((pad_length,), self.pad_token_id, dtype=input_id.dtype)
                ])
                padded_label = torch.cat([
                    label,
                    torch.full((pad_length,), -100, dtype=label.dtype)
                ])
                attention_mask = torch.cat([
                    torch.ones(current_len, dtype=torch.long),
                    torch.zeros(pad_length, dtype=torch.long)
                ])
            else:
                # Truncate if too long
                padded_input_id = input_id[:max_len]
                padded_label = label[:max_len]
                attention_mask = torch.ones(len(padded_input_id), dtype=torch.long)
            
            padded_input_ids.append(padded_input_id)
            padded_labels.append(padded_label)
            padded_attention_masks.append(attention_mask)
        
        # Final safety check
        if not padded_input_ids:
            raise ValueError("No valid padded sequences created")
        
        # Stack all tensors
        try:
            batch['input_ids'] = torch.stack(padded_input_ids)
            batch['labels'] = torch.stack(padded_labels)
            batch['attention_mask'] = torch.stack(padded_attention_masks)
        except Exception as e:
            print(f"Error stacking tensors: {e}")
            print(f"Shapes: input_ids={[x.shape for x in padded_input_ids[:3]]}")
            raise e
        
        return batch

def setup_training_args(output_dir="./checkpoints/sophiavl_lora", num_epochs=3, batch_size=1):
    """Setup training arguments optimized for LoRA fine-tuning"""
    
    return TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,  # Smaller batch size for stability
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=16,  # Increase to maintain effective batch size
        warmup_steps=50,  # Reduced warmup steps
        learning_rate=5e-4,  # Higher LR for LoRA
        weight_decay=0.01,
        logging_dir=f"./logs/sophiavl_lora_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
        logging_steps=10,
        save_steps=100,  # More frequent saves
        save_total_limit=3,
        eval_strategy="steps",  # Updated parameter name
        eval_steps=100,  # More frequent evaluation
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        fp16=True,  # Use mixed precision
        dataloader_pin_memory=False,  # Disable for memory efficiency
        remove_unused_columns=False,  # Keep all columns for vision models
        report_to=[],  # Disable external reporting to avoid issues
        seed=42,
        data_seed=42,
        group_by_length=False,  # Disable for vision models
        dataloader_num_workers=0,  # Single-threaded to avoid issues
        max_grad_norm=1.0,  # Gradient clipping for stability
    )

def train_sophiavl_lora():
    """Main training function for SophiaVL LoRA"""
    
    print("="*60)
    print("STARTING SOPHIAVL-R1 LORA TRAINING")
    print("="*60)
    
    # Get training data with validation
    training_data = get_training_data()
    if not training_data:
        raise ValueError("No training data available!")
    
    print(f"Training on {len(training_data)} examples from testmini split")
    
    # Create dataset with more robust error handling
    print("Creating dataset...")
    
    # Filter out problematic examples first
    print("Filtering training data...")
    filtered_data = []
    for i, example in enumerate(training_data):
        try:
            # Basic validation
            if (example.get('image') is not None and 
                example.get('formatted_question') and 
                len(example['formatted_question']) < 2000):  # Length limit
                filtered_data.append(example)
            else:
                print(f"Skipping example {i}: missing data or too long")
        except Exception as e:
            print(f"Error checking example {i}: {e}")
            continue
    
    print(f"Filtered data: {len(filtered_data)} examples (from {len(training_data)})")
    
    if len(filtered_data) < 10:
        raise ValueError("Too few valid training examples after filtering!")
    
    # Split data for validation (use 10% for validation)
    split_idx = int(0.9 * len(filtered_data))
    train_split = filtered_data[:split_idx]
    val_split = filtered_data[split_idx:] if split_idx < len(filtered_data) else filtered_data[-10:]  # At least 10 for validation
    
    print(f"Train split: {len(train_split)} examples")
    print(f"Validation split: {len(val_split)} examples")
    
    # Create datasets with reduced max_length
    train_dataset = MathVistaDataset(train_split, model_processor, max_length=768)
    val_dataset = MathVistaDataset(val_split, model_processor, max_length=768)
    
    print(f"Train dataset size: {len(train_dataset)}")
    print(f"Validation dataset size: {len(val_dataset)}")
    
    # Test the dataset and collator before training
    print("Testing data loading...")
    valid_items = []
    for i in range(min(10, len(train_dataset))):
        try:
            item = train_dataset[i]
            if item is not None and 'input_ids' in item:
                valid_items.append(item)
                if len(valid_items) >= 2:
                    break
        except Exception as e:
            print(f"Error loading item {i}: {e}")
            continue
    
    if len(valid_items) < 2:
        raise ValueError("Could not load enough valid training examples")
    
    print(f"✅ Loaded {len(valid_items)} valid items for testing")
    
    # Test collator
    data_collator = CustomDataCollator(model_processor)
    test_batch = data_collator(valid_items[:2])
    print(f"✅ Data collator working: {test_batch['input_ids'].shape}")
    
    # Setup training arguments with more conservative settings
    training_args = setup_training_args(
        output_dir="./checkpoints/sophiavl_lora",
        num_epochs=1,  # Start with just 1 epoch
        batch_size=1   # Keep batch size small
    )
    
    # Create trainer
    print("Initializing trainer...")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        tokenizer=model_processor.tokenizer,  # Pass tokenizer for saving
    )
    
    # Print model info before training
    print("\nModel configuration:")
    print(f"Model type: {type(model).__name__}")
    if hasattr(model, 'print_trainable_parameters'):
        model.print_trainable_parameters()
    
    # Start training
    print("\nStarting training...")
    try:
        # Clear memory before training
        clear_memory()
        
        # Train the model
        train_result = trainer.train()
        
        # Check if training result is valid
        if train_result is None:
            raise ValueError("Training returned None result")
        
        # Save the final model
        print("\nSaving trained model...")
        trainer.save_model()
        trainer.save_state()
        
        # Print training results
        print("\n" + "="*60)
        print("TRAINING COMPLETED SUCCESSFULLY!")
        print("="*60)
        print(f"Final training loss: {getattr(train_result, 'training_loss', 'N/A')}")
        
        if hasattr(train_result, 'metrics') and train_result.metrics:
            print(f"Training time: {train_result.metrics.get('train_runtime', 'N/A')} seconds")
            print(f"Samples per second: {train_result.metrics.get('train_samples_per_second', 'N/A')}")
            
            # Save training metrics
            with open(f"{training_args.output_dir}/training_metrics.json", "w") as f:
                json.dump(train_result.metrics, f, indent=2)
        
        return trainer, train_result
        
    except Exception as e:
        print(f"\nTraining failed with error: {e}")
        import traceback
        traceback.print_exc()
        
        print("Attempting to save checkpoint...")
        try:
            if 'trainer' in locals():
                trainer.save_model(f"{training_args.output_dir}/emergency_checkpoint")
                print("Emergency checkpoint saved!")
            else:
                print("Trainer not initialized, cannot save checkpoint")
        except Exception as save_error:
            print(f"Could not save emergency checkpoint: {save_error}")
        
        raise e

# Test a single example first to ensure everything works
def test_single_example():
    """Test processing a single example to debug any issues"""
    print("Testing single example processing...")
    
    training_data = get_training_data()
    if not training_data:
        print("No training data available!")
        return False
    
    try:
        # Test dataset creation
        test_dataset = MathVistaDataset([training_data[0]], model_processor)
        test_item = test_dataset[0]
        
        print("Single example processed successfully!")
        print(f"Input shape: {test_item['input_ids'].shape}")
        print(f"Labels shape: {test_item['labels'].shape}")
        if 'pixel_values' in test_item:
            print(f"Pixel values shape: {test_item['pixel_values'].shape}")
        
        # Test data collator
        collator = CustomDataCollator(model_processor)
        batch = collator([test_item])
        
        print("Data collation successful!")
        print(f"Batch input_ids shape: {batch['input_ids'].shape}")
        print(f"Batch labels shape: {batch['labels'].shape}")
        
        return True
        
    except Exception as e:
        print(f"Error in single example test: {e}")
        return False

# Run the test first
if test_single_example():
    print("\n✅ Single example test passed! Ready to start training.")
    
    # Ask user confirmation before starting full training
    print("\n" + "="*60)
    print("READY TO START TRAINING")
    print("="*60)
    print("This will fine-tune the SophiaVL-R1 model with LoRA on the MathVista testmini dataset.")
    print("Training will take several hours depending on your hardware.")
    print("\nTo start training, run the next cell or call: train_sophiavl_lora()")
else:
    print("\n❌ Single example test failed. Please check the error messages above.")

# Store training function for easy access
training_functions = ['train_sophiavl_lora', 'test_single_example', 'setup_training_args']
print(f"\nAvailable training functions: {training_functions}")

Testing single example processing...
Pre-processing 1 examples...
Dataset initialized with 0 valid examples from 1 raw examples
Error in single example test: Index 0 out of range for dataset of size 0

❌ Single example test failed. Please check the error messages above.

Available training functions: ['train_sophiavl_lora', 'test_single_example', 'setup_training_args']


In [37]:
# Test Updated Dataset and Collator - Fresh Version
# Run this cell to test the improved data processing before full training

print("🔧 Testing Updated Dataset and Data Collator (Fresh)...")

try:
    # Get training data
    training_data = get_training_data()
    if not training_data:
        print("❌ No training data available")
    else:
        print(f"✅ Got {len(training_data)} training examples")
        
        # Filter problematic examples like in the training function
        filtered_data = []
        for i, example in enumerate(training_data[:10]):  # Test first 10
            try:
                if (example.get('image') is not None and 
                    example.get('formatted_question') and 
                    len(example['formatted_question']) < 2000):
                    filtered_data.append(example)
                else:
                    print(f"Skipping example {i}: missing data or too long")
            except Exception as e:
                print(f"Error checking example {i}: {e}")
                continue
        
        print(f"Filtered {len(filtered_data)} valid examples from first 10")
        
        if len(filtered_data) < 2:
            print("❌ Not enough valid examples for testing")
        else:
            # Create fresh test dataset with the new implementation
            test_dataset = MathVistaDataset(filtered_data, model_processor, max_length=768)
            
            # Test individual items
            print("\nTesting individual items...")
            valid_items = []
            for i in range(min(3, len(test_dataset))):
                try:
                    item = test_dataset[i]
                    print(f"✅ Item {i}: input_ids={item['input_ids'].shape}, labels={item['labels'].shape}")
                    if 'pixel_values' in item:
                        print(f"   pixel_values={item['pixel_values'].shape}")
                    valid_items.append(item)
                except Exception as e:
                    print(f"❌ Item {i} failed: {e}")
            
            if len(valid_items) >= 2:
                # Create a fresh data collator instance
                print("\nTesting fresh data collator...")
                fresh_data_collator = CustomDataCollator(model_processor)
                
                # Test with 2 items that have different pixel_values shapes
                print("Testing with items that have different pixel_values shapes...")
                test_items = valid_items[:2]
                
                batch = fresh_data_collator(test_items)
                
                print(f"✅ Batch created successfully!")
                print(f"  Batch input_ids shape: {batch['input_ids'].shape}")
                print(f"  Batch labels shape: {batch['labels'].shape}")
                print(f"  Batch attention_mask shape: {batch['attention_mask'].shape}")
                
                if 'pixel_values' in batch:
                    print(f"  Batch pixel_values shape: {batch['pixel_values'].shape}")
                
                # Test with different batch sizes
                print("\nTesting different batch sizes...")
                for batch_size in [1, 2, 3]:
                    if len(valid_items) >= batch_size:
                        test_batch = fresh_data_collator(valid_items[:batch_size])
                        print(f"  Batch size {batch_size}: ✅ input_ids={test_batch['input_ids'].shape}, pixel_values={test_batch['pixel_values'].shape}")
                
                print("\n🎉 All tests passed! Data processing is working correctly.")
                print("The pixel_values padding is working properly.")
                print("You can now run the training with confidence.")
                
            else:
                print("❌ Not enough valid items for batch testing")
        
except Exception as e:
    print(f"❌ Testing failed: {e}")
    import traceback
    traceback.print_exc()
    print("Please check the error above before proceeding with training.")

🔧 Testing Updated Dataset and Data Collator (Fresh)...
✅ Got 1000 training examples
Filtered 10 valid examples from first 10
Pre-processing 10 examples...
Dataset initialized with 0 valid examples from 10 raw examples

Testing individual items...
❌ Not enough valid items for batch testing


In [38]:
# Proper Vision-Language Dataset with Real Image Processing
# This implements proper image processing using the actual MathVista images

class ProperVisionLanguageDataset(Dataset):
    """Proper dataset that uses actual images with SophiaVL-R1 processor"""
    
    def __init__(self, processed_data, model_processor, max_length=1024, max_examples=100):
        self.raw_data = processed_data[:max_examples]  # Limit for memory
        self.model_processor = model_processor
        self.max_length = max_length
        
        print(f"🔄 Processing {len(self.raw_data)} examples with proper image handling...")
        
        # Pre-process and cache valid examples
        self.valid_examples = []
        successful_count = 0
        
        for idx, example in enumerate(tqdm(self.raw_data, desc="Processing examples")):
            try:
                processed_item = self._process_example_properly(idx, example)
                if processed_item is not None:
                    self.valid_examples.append(processed_item)
                    successful_count += 1
            except Exception as e:
                if idx < 5:  # Only show first few errors
                    print(f"Warning: Example {idx} failed: {e}")
                continue
        
        print(f"✅ Successfully processed {successful_count} examples out of {len(self.raw_data)}")
        
        if len(self.valid_examples) < 5:
            raise ValueError("Too few valid examples processed. Check your data and processor.")
    
    def _process_example_properly(self, idx, example):
        """Process example using proper SophiaVL-R1 format"""
        try:
            # Get the actual image from the dataset
            image = example['image']  # This should be a PIL Image
            question = example['formatted_question']
            answer = str(example['ground_truth_answer'])
            
            # Truncate question if too long
            if len(question) > 800:
                question = question[:800] + "..."
            
            # Create the conversation format that SophiaVL expects
            # Based on the processor, we need to format it properly
            conversation = [
                {
                    "role": "user", 
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": question}
                    ]
                },
                {
                    "role": "assistant",
                    "content": answer
                }
            ]
            
            # Apply chat template to get formatted text
            formatted_text = self.model_processor.apply_chat_template(
                conversation,
                tokenize=False,
                add_generation_prompt=False
            )
            
            # Now process with both text and image
            inputs = self.model_processor(
                text=formatted_text,
                images=image,
                return_tensors="pt",
                padding=False,
                truncation=True,
                max_length=self.max_length
            )
            
            # Flatten batch dimensions
            for key in inputs:
                if isinstance(inputs[key], torch.Tensor) and inputs[key].dim() > 1:
                    # Keep proper dimensions for pixel_values and image_grid_thw
                    if key in ['pixel_values', 'image_grid_thw']:
                        if inputs[key].shape[0] == 1:  # If batch size is 1, squeeze it
                            inputs[key] = inputs[key].squeeze(0)
                    else:
                        inputs[key] = inputs[key].squeeze(0)
            
            # Create labels for training
            inputs['labels'] = inputs['input_ids'].clone()
            
            # Add metadata
            inputs['ground_truth'] = answer
            inputs['question_id'] = example.get('pid', f'example_{idx}')
            
            # Debug first few examples
            if idx < 3:
                print(f"✅ Example {idx} processed:")
                print(f"   input_ids: {inputs['input_ids'].shape}")
                print(f"   pixel_values: {inputs['pixel_values'].shape}")
                print(f"   image_grid_thw: {inputs['image_grid_thw'].shape}")
                print(f"   Question length: {len(question)} chars")
                print(f"   Answer: {answer[:50]}...")
            
            return inputs
            
        except Exception as e:
            # If processing fails, return None to skip this example
            return None
    
    def __len__(self):
        return len(self.valid_examples)
    
    def __getitem__(self, idx):
        if idx >= len(self.valid_examples):
            raise IndexError(f"Index {idx} out of range")
        return self.valid_examples[idx]

class ProperVisionLanguageCollator:
    """Data collator that properly handles vision-language model inputs"""
    
    def __init__(self, model_processor):
        self.model_processor = model_processor
        self.tokenizer = model_processor.tokenizer
        self.pad_token_id = self.tokenizer.pad_token_id or self.tokenizer.eos_token_id
    
    def __call__(self, features):
        if not features:
            raise ValueError("Empty features list")
        
        # Filter out None features
        features = [f for f in features if f is not None]
        if not features:
            raise ValueError("All features are None")
        
        batch = {}
        
        # Handle text tokens
        input_ids = [f['input_ids'] for f in features]
        labels = [f['labels'] for f in features]
        
        # Pad text sequences
        max_length = max(len(ids) for ids in input_ids)
        
        padded_input_ids = []
        padded_labels = []
        attention_masks = []
        
        for input_id, label in zip(input_ids, labels):
            # Ensure 1D tensors
            if input_id.dim() > 1:
                input_id = input_id.squeeze()
            if label.dim() > 1:
                label = label.squeeze()
            
            current_length = len(input_id)
            pad_length = max_length - current_length
            
            if pad_length > 0:
                # Pad sequences
                padded_input_id = torch.cat([
                    input_id,
                    torch.full((pad_length,), self.pad_token_id, dtype=input_id.dtype)
                ])
                padded_label = torch.cat([
                    label,
                    torch.full((pad_length,), -100, dtype=label.dtype)  # -100 for ignored tokens
                ])
                attention_mask = torch.cat([
                    torch.ones(current_length, dtype=torch.long),
                    torch.zeros(pad_length, dtype=torch.long)
                ])
            else:
                padded_input_id = input_id
                padded_label = label
                attention_mask = torch.ones(current_length, dtype=torch.long)
            
            padded_input_ids.append(padded_input_id)
            padded_labels.append(padded_label)
            attention_masks.append(attention_mask)
        
        # Stack text tensors
        batch['input_ids'] = torch.stack(padded_input_ids)
        batch['labels'] = torch.stack(padded_labels)
        batch['attention_mask'] = torch.stack(attention_masks)
        
        # Handle vision inputs
        pixel_values = [f['pixel_values'] for f in features]
        image_grid_thw = [f['image_grid_thw'] for f in features]
        
        # Check if all pixel_values have the same shape
        pixel_shapes = [pv.shape for pv in pixel_values]
        if len(set(pixel_shapes)) == 1:
            # All same shape - can stack directly
            batch['pixel_values'] = torch.stack(pixel_values)
        else:
            # Different shapes - need to pad
            max_dim0 = max(pv.shape[0] for pv in pixel_values)
            padded_pixel_values = []
            
            for pv in pixel_values:
                if pv.shape[0] < max_dim0:
                    # Pad the first dimension
                    pad_size = max_dim0 - pv.shape[0]
                    padding = torch.zeros(pad_size, pv.shape[1], dtype=pv.dtype, device=pv.device)
                    padded_pv = torch.cat([pv, padding], dim=0)
                else:
                    padded_pv = pv
                padded_pixel_values.append(padded_pv)
            
            batch['pixel_values'] = torch.stack(padded_pixel_values)
        
        # Handle image_grid_thw
        batch['image_grid_thw'] = torch.stack(image_grid_thw)
        
        return batch

def create_proper_training_setup():
    """Create a proper training setup with real image processing"""
    print("🚀 Setting up proper vision-language training...")
    
    # Get training data
    training_data = get_training_data()
    if not training_data:
        raise ValueError("No training data available!")
    
    print(f"📊 Available training examples: {len(training_data)}")
    
    # Create datasets with proper image processing
    print("🔄 Creating vision-language datasets...")
    
    # Use a reasonable subset for training and validation
    train_size = min(80, len(training_data) - 20)  # Leave 20 for validation
    val_size = min(20, len(training_data) - train_size)
    
    train_data = training_data[:train_size]
    val_data = training_data[train_size:train_size + val_size]
    
    # Create datasets
    train_dataset = ProperVisionLanguageDataset(
        train_data, 
        model_processor, 
        max_length=512,  # Conservative length
        max_examples=train_size
    )
    
    val_dataset = ProperVisionLanguageDataset(
        val_data, 
        model_processor, 
        max_length=512,
        max_examples=val_size
    )
    
    print(f"✅ Training dataset: {len(train_dataset)} examples")
    print(f"✅ Validation dataset: {len(val_dataset)} examples")
    
    # Create data collator
    data_collator = ProperVisionLanguageCollator(model_processor)
    
    # Test the setup
    print("🧪 Testing the setup...")
    test_item_0 = train_dataset[0]
    test_item_1 = train_dataset[1] if len(train_dataset) > 1 else train_dataset[0]
    
    test_batch = data_collator([test_item_0, test_item_1])
    
    print("✅ Setup test successful!")
    print(f"   Batch input_ids: {test_batch['input_ids'].shape}")
    print(f"   Batch pixel_values: {test_batch['pixel_values'].shape}")
    print(f"   Batch image_grid_thw: {test_batch['image_grid_thw'].shape}")
    
    return train_dataset, val_dataset, data_collator

def train_proper_vision_language():
    """Train SophiaVL-R1 with proper vision-language processing"""
    print("="*60)
    print("🎯 PROPER SOPHIAVL-R1 VISION-LANGUAGE TRAINING")
    print("="*60)
    
    try:
        # Create proper datasets
        train_dataset, val_dataset, data_collator = create_proper_training_setup()
        
        # Setup training arguments
        training_args = TrainingArguments(
            output_dir="./checkpoints/sophiavl_proper",
            num_train_epochs=2,  # Start with 2 epochs
            per_device_train_batch_size=1,  # Small batch for stability
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=8,  # Effective batch size of 8
            warmup_steps=10,
            learning_rate=5e-4,
            weight_decay=0.01,
            logging_dir="./logs/sophiavl_proper",
            logging_steps=5,
            save_steps=50,
            save_total_limit=2,
            eval_strategy="steps",
            eval_steps=25,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            fp16=True,
            remove_unused_columns=False,
            report_to=[],  # Disable wandb
            dataloader_num_workers=0,
            dataloader_pin_memory=False,
            max_grad_norm=1.0,
            seed=42,
        )
        
        # Create trainer
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            data_collator=data_collator,
            processing_class=model_processor,
        )
        
        print("📈 Model info:")
        if hasattr(model, 'print_trainable_parameters'):
            model.print_trainable_parameters()
        
        # Clear memory and start training
        clear_memory()
        
        print("🚀 Starting training...")
        train_result = trainer.train()
        
        # Save the model
        print("💾 Saving trained model...")
        trainer.save_model()
        trainer.save_state()
        
        print("🎉 Training completed successfully!")
        print(f"✅ Model saved to: {training_args.output_dir}")
        
        return trainer, train_result
        
    except Exception as e:
        print(f"❌ Training failed: {e}")
        import traceback
        traceback.print_exc()
        raise e

print("🎯 Proper vision-language training setup loaded!")
print("📝 Available functions:")
print("   - create_proper_training_setup(): Setup datasets and collator")
print("   - train_proper_vision_language(): Start full training")
print("")
print("💡 To start training with proper image processing:")
print("   trainer, result = train_proper_vision_language()")

🎯 Proper vision-language training setup loaded!
📝 Available functions:
   - create_proper_training_setup(): Setup datasets and collator
   - train_proper_vision_language(): Start full training

💡 To start training with proper image processing:
   trainer, result = train_proper_vision_language()


In [39]:
# Start Proper Vision-Language Training
print("🚀 Starting SophiaVL-R1 training with proper image processing...")

try:
    trainer, result = train_proper_vision_language()
    
    print("\n🎉 Training completed successfully!")
    print("✅ Model with proper image processing trained!")
    print("✅ Ready for evaluation on test set")
    
except Exception as e:
    print(f"\n❌ Training failed: {e}")
    print("Check the error messages above for debugging")

🚀 Starting SophiaVL-R1 training with proper image processing...
🎯 PROPER SOPHIAVL-R1 VISION-LANGUAGE TRAINING
🚀 Setting up proper vision-language training...
📊 Available training examples: 1000
🔄 Creating vision-language datasets...
🔄 Processing 80 examples with proper image handling...


Processing examples:   0%|          | 0/80 [00:00<?, ?it/s]

✅ Example 2 processed:
   input_ids: torch.Size([212])
   pixel_values: torch.Size([40, 1176])
   image_grid_thw: torch.Size([3])
   Question length: 334 chars
   Answer: 145°...
✅ Successfully processed 66 examples out of 80
🔄 Processing 20 examples with proper image handling...


Processing examples:   0%|          | 0/20 [00:00<?, ?it/s]

✅ Example 0 processed:
   input_ids: torch.Size([450])
   pixel_values: torch.Size([1408, 1176])
   image_grid_thw: torch.Size([3])
   Question length: 270 chars
   Answer: -1...
✅ Example 1 processed:
   input_ids: torch.Size([229])
   pixel_values: torch.Size([196, 1176])
   image_grid_thw: torch.Size([3])
   Question length: 472 chars
   Answer: 6...
✅ Example 2 processed:
   input_ids: torch.Size([503])
   pixel_values: torch.Size([1564, 1176])
   image_grid_thw: torch.Size([3])
   Question length: 346 chars
   Answer: Yes...
✅ Successfully processed 15 examples out of 20
✅ Training dataset: 66 examples
✅ Validation dataset: 15 examples
🧪 Testing the setup...
✅ Setup test successful!
   Batch input_ids: torch.Size([2, 212])
   Batch pixel_values: torch.Size([2, 396, 1176])
   Batch image_grid_thw: torch.Size([2, 3])


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


📈 Model info:
trainable params: 3,686,400 || all params: 3,758,309,376 || trainable%: 0.0981
Cleared memory. Current CPU memory usage: 21308.14 MB, GPU memory usage: 0.00 MB
🚀 Starting training...


Step,Training Loss,Validation Loss


💾 Saving trained model...
🎉 Training completed successfully!
✅ Model saved to: ./checkpoints/sophiavl_proper

🎉 Training completed successfully!
✅ Model with proper image processing trained!
✅ Ready for evaluation on test set


In [40]:
# Test the Trained Model
print("🧪 Testing the trained SophiaVL-R1 model...")

try:
    accuracy, results = evaluate_trained_sophiavl(num_samples=15)
    
    if accuracy is not None:
        print(f"\n🎉 Evaluation completed!")
        print(f"🎯 Final accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    else:
        print("❌ Evaluation failed")
        
except Exception as e:
    print(f"❌ Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

🧪 Testing the trained SophiaVL-R1 model...
❌ Error during evaluation: name 'evaluate_trained_sophiavl' is not defined


Traceback (most recent call last):
  File "/tmp/ipykernel_47938/281891327.py", line 5, in <module>
    accuracy, results = evaluate_trained_sophiavl(num_samples=15)
NameError: name 'evaluate_trained_sophiavl' is not defined


In [41]:
# Fix device placement - move base model to cuda:1 to match training device
print("Moving base model to cuda:1...")
model = model.to(device)
print(f"Base model is now on: {next(model.parameters()).device}")

# Verify all components are on the same device
print(f"Device consistency check:")
print(f"  Device variable: {device}")
print(f"  Base model device: {next(model.parameters()).device}")
print(f"  Match: {device == next(model.parameters()).device}")

Moving base model to cuda:1...
Base model is now on: cuda:1
Device consistency check:
  Device variable: cuda:1
  Base model device: cuda:1
  Match: True


In [42]:
# Debug the ground truth issue
print("🔍 Debugging ground truth data...")
test_data = get_evaluation_data()[:5]

for i, example in enumerate(test_data):
    print(f"\nExample {i+1}:")
    print(f"  Keys in example: {list(example.keys())}")
    print(f"  Ground truth answer: '{example.get('ground_truth_answer', 'KEY_NOT_FOUND')}'")
    print(f"  Answer key: '{example.get('answer', 'KEY_NOT_FOUND')}'")
    print(f"  Expected answer: '{example.get('expected_answer', 'KEY_NOT_FOUND')}'")
    print(f"  All answer fields:")
    for key in example.keys():
        if 'answer' in key.lower() or 'truth' in key.lower():
            print(f"    {key}: '{example[key]}'")
    if i >= 2:  # Just show first 3
        break

🔍 Debugging ground truth data...

Example 1:
  Keys in example: ['pid', 'formatted_question', 'original_question', 'image', 'ground_truth_answer', 'question_type', 'answer_type', 'choices', 'unit', 'precision', 'metadata', 'query']
  Ground truth answer: ''
  Answer key: 'KEY_NOT_FOUND'
  Expected answer: 'KEY_NOT_FOUND'
  All answer fields:
    ground_truth_answer: ''
    answer_type: 'integer'

Example 2:
  Keys in example: ['pid', 'formatted_question', 'original_question', 'image', 'ground_truth_answer', 'question_type', 'answer_type', 'choices', 'unit', 'precision', 'metadata', 'query']
  Ground truth answer: ''
  Answer key: 'KEY_NOT_FOUND'
  Expected answer: 'KEY_NOT_FOUND'
  All answer fields:
    ground_truth_answer: ''
    answer_type: 'text'

Example 3:
  Keys in example: ['pid', 'formatted_question', 'original_question', 'image', 'ground_truth_answer', 'question_type', 'answer_type', 'choices', 'unit', 'precision', 'metadata', 'query']
  Ground truth answer: ''
  Answer key:

In [ ]:
# Fix: Create a simple evaluation function that uses the original dataset
def get_fixed_evaluation_data(num_samples=20):
    """Get evaluation data with proper ground truth answers"""
    # Use the original dataset directly
    raw_test_data = list(dataset['testmini'])[:num_samples]
    
    fixed_data = []
    for item in raw_test_data:
        # Create a fixed example with proper ground truth
        fixed_example = {
            'pid': item['pid'],
            'image': item['decoded_image'],
            'formatted_question': item['question'],
            'original_question': item['question'],
            'ground_truth_answer': item['answer'],  # This is the key fix!
            'question_type': item['question_type'],
            'answer_type': item['answer_type'],
            'choices': item['choices'],
            'unit': item['unit'],
            'precision': item['precision'],
            'metadata': item['metadata'],
            'query': item['query']
        }
        fixed_data.append(fixed_example)
    
    return fixed_data

# Test the fixed data
print("🔧 Testing fixed evaluation data...")
fixed_test_data = get_fixed_evaluation_data(5)

for i, example in enumerate(fixed_test_data[:3]):
    print(f"\nFixed Example {i+1}:")
    print(f"  PID: {example['pid']}")
    print(f"  Ground truth: '{example['ground_truth_answer']}'")
    print(f"  Answer type: {example['answer_type']}")
    print(f"  Question: {example['formatted_question'][:50]}...")

print(f"\n✅ Fixed evaluation data ready with {len(fixed_test_data)} examples")

🔧 Testing fixed evaluation data...

Fixed Example 1:
  PID: 1
  Ground truth: '1.2'
  Answer type: float
  Question: When a spring does work on an object, we cannot fi...

Fixed Example 2:
  PID: 2
  Ground truth: '1000'
  Answer type: integer
  Question: what is the total volume of the measuring cup?...

Fixed Example 3:
  PID: 3
  Ground truth: '145°'
  Answer type: text
  Question: △ABC的两内角平分线OB、OC相交于点O，若∠A＝110°，则∠BOC＝（）...

✅ Fixed evaluation data ready with 5 examples


: 

In [ ]:
def evaluate_trained_sophiavl_fixed(model_path="./checkpoints/sophiavl_proper", num_samples=20):
    """Evaluate the trained SophiaVL-R1 model with fixed ground truth data"""
    print("🔍 Evaluating trained SophiaVL-R1 model with fixed data...")
    
    try:
        from peft import PeftModel
        
        # Load the trained LoRA model
        print(f"📂 Loading model from {model_path}")
        trained_model = PeftModel.from_pretrained(model, model_path)
        trained_model = trained_model.to(device)  # Ensure model is on correct device
        trained_model.eval()
        
        # Get test data with proper ground truth
        test_data = get_fixed_evaluation_data(num_samples)
        print(f"📊 Evaluating on {len(test_data)} test samples...")
        
        correct_predictions = 0
        total_predictions = 0
        results = []
        
        with torch.no_grad():
            for i, example in enumerate(tqdm(test_data, desc="Evaluating")):
                try:
                    # Prepare the conversation format
                    conversation = [
                        {
                            "role": "user",
                            "content": [
                                {"type": "image"},
                                {"type": "text", "text": example['formatted_question']}
                            ]
                        }
                    ]
                    
                    # Apply chat template
                    text = model_processor.apply_chat_template(
                        conversation, 
                        tokenize=False, 
                        add_generation_prompt=True
                    )
                    
                    # Process inputs
                    inputs = model_processor(
                        text=text,
                        images=example['image'],
                        return_tensors="pt"
                    )
                    
                    # Move all inputs to the correct device
                    for key in inputs:
                        if isinstance(inputs[key], torch.Tensor):
                            inputs[key] = inputs[key].to(device)
                    
                    # Generate response
                    with torch.amp.autocast('cuda'):  # Updated autocast syntax
                        generated_ids = trained_model.generate(
                            **inputs,
                            max_new_tokens=50,
                            do_sample=False,
                            temperature=0.0,
                            pad_token_id=model_processor.tokenizer.eos_token_id
                        )
                    
                    # Decode response
                    generated_text = model_processor.batch_decode(
                        generated_ids[:, inputs['input_ids'].shape[1]:], 
                        skip_special_tokens=True
                    )[0].strip()
                    
                    # Check accuracy with proper ground truth
                    ground_truth = str(example['ground_truth_answer']).strip()
                    prediction = generated_text.strip()
                    
                    is_correct = prediction.lower() == ground_truth.lower()
                    if is_correct:
                        correct_predictions += 1
                    
                    total_predictions += 1
                    
                    # Store result
                    results.append({
                        'question_id': example.get('pid', f'test_{i}'),
                        'question': example['formatted_question'][:100] + "..." if len(example['formatted_question']) > 100 else example['formatted_question'],
                        'ground_truth': ground_truth,
                        'prediction': prediction,
                        'correct': is_correct
                    })
                    
                    # Print all examples for small test
                    print(f"\n📝 Example {i+1}:")
                    print(f"   Question: {example['formatted_question'][:100]}...")
                    print(f"   Ground Truth: '{ground_truth}'")
                    print(f"   Prediction: '{prediction}'")
                    print(f"   {'✅ Correct' if is_correct else '❌ Incorrect'}")
                
                except Exception as e:
                    print(f"⚠️ Error evaluating example {i}: {e}")
                    continue
        
        # Calculate final accuracy
        accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
        
        print(f"\n{'='*50}")
        print(f"🎯 EVALUATION RESULTS (FIXED)")
        print(f"{'='*50}")
        print(f"📊 Total samples evaluated: {total_predictions}")
        print(f"✅ Correct predictions: {correct_predictions}")
        print(f"❌ Incorrect predictions: {total_predictions - correct_predictions}")
        print(f"🎯 Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
        
        return accuracy, results
        
    except Exception as e:
        print(f"❌ Evaluation failed: {e}")
        import traceback
        traceback.print_exc()
        return None, None

print("✅ Fixed evaluation function loaded!")

In [ ]:
# Run a larger evaluation to get better statistics
print("🔍 Running larger evaluation...")
accuracy_20, results_20 = evaluate_trained_sophiavl_fixed(num_samples=20)

if accuracy_20 is not None:
    print(f"\n🎯 FINAL RESULTS (20 samples):")
    print(f"   Accuracy: {accuracy_20:.4f} ({accuracy_20*100:.2f}%)")
    
    # Show some examples of different types
    correct_examples = [r for r in results_20 if r['correct']]
    incorrect_examples = [r for r in results_20 if not r['correct']]
    
    print(f"\n✅ Correct predictions: {len(correct_examples)}")
    if correct_examples:
        for i, ex in enumerate(correct_examples[:3]):
            print(f"   {i+1}. Q: {ex['question'][:50]}...")
            print(f"      GT: '{ex['ground_truth']}' | Pred: '{ex['prediction']}'")
    
    print(f"\n❌ Incorrect predictions: {len(incorrect_examples)}")
    if incorrect_examples:
        for i, ex in enumerate(incorrect_examples[:3]):
            print(f"   {i+1}. Q: {ex['question'][:50]}...")
            print(f"      GT: '{ex['ground_truth']}' | Pred: '{ex['prediction'][:50]}...'")
else:
    print("❌ Large evaluation failed")